# TruthLens: Debiased, Explainable Fake News Detection

**Course:** C403 - Machine Learning Mini Project  
**Team:** Riddhi Patil (52), Prashant Jha (54), Aditya Soni (57), Riwan Pereira (65)  
**Guide:** Dr. Joanne Gomes | **Institute:** St. Francis Institute of Technology

---

## 7 Critical Mistakes in Existing Models (and Our Solutions)

| # | Problem | Our Solution |
|---|---|---|
| 1 | Dataset Bias (models learn "Reuters = Real") | Entity masking + source name stripping |
| 2 | Generalization Failure (99% ISOT, 45% LIAR) | Cross-dataset evaluation |
| 3 | TF-IDF can't capture semantics | Hybrid: TF-IDF + GloVe embeddings |
| 4 | Ignoring writing style context | Stylometric features (sentiment, readability) |
| 5 | AI-generated text undetectable | Perplexity, burstiness, type-token ratio |
| 6 | Black box models | LIME + SHAP explainability + bias auditing |
| 7 | Majority voting is naive | Stacking ensemble with LR meta-learner |

## 1. Imports & Configuration

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Ensure TruthLens src is importable
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.abspath(''))

import config

print(f"Random State: {config.RANDOM_STATE}")
print(f"Test Size: {config.TEST_SIZE}")
print(f"CV Folds: {config.CV_FOLDS}")
print(f"SVD Components: {config.SVD_COMPONENTS}")
print(f"Entity Masking: {config.ENABLE_ENTITY_MASKING}")
print(f"GloVe: {config.ENABLE_GLOVE}")

## 2. Data Loading

We use two datasets:
- **ISOT Fake News Dataset**: ~44,898 full-length news articles (Reuters = Real, various = Fake)
- **LIAR Dataset**: ~12,836 short political statements from PolitiFact (6-class mapped to binary)

In [ ]:
from src.data_loader import get_datasets, split_dataset

datasets = get_datasets(download=True)

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"Dataset: {name.upper()}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Label distribution:\n{df['label'].value_counts()}")
    print(f"\nSample text (first 200 chars): {str(df['text'].iloc[0])[:200]}...")

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, name in enumerate(['isot', 'liar']):
    if name in datasets:
        df = datasets[name]
        counts = df['label'].value_counts()
        labels = ['Real (0)', 'Fake (1)']
        colors = ['#27ae60', '#e74c3c']
        axes[i].bar(labels, [counts.get(0, 0), counts.get(1, 0)], color=colors)
        axes[i].set_title(f'{name.upper()} Dataset')
        axes[i].set_ylabel('Count')

plt.suptitle('Label Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Preprocessing

Our 6-step pipeline:
1. **Reuters/AP byline stripping** (removes source identity leakage)
2. **Lowercasing** + URL/HTML removal
3. **Entity masking** (spaCy NER: PERSON, ORG, GPE, LOC, DATE -> tokens)
4. **Supplementary source masking** (regex for Reuters, CNN, Fox News, etc.)
5. **Stopword removal** (NLTK)
6. **Lemmatization** (spaCy)

In [ ]:
from src.preprocessor import preprocess_dataframe, load_processed, save_processed, clean_text, mask_entities

# Demo: show preprocessing on sample texts
demo_texts = [
    "WASHINGTON (Reuters) - Donald Trump announced new policies at the White House yesterday.",
    "BREAKING: Scientists at Google discovered a cure for cancer!!! Click here to learn more!!!",
    "The economy is growing steadily according to recent Federal Reserve reports from CNN.",
]

print("=== Preprocessing Demo ===")
for text in demo_texts:
    cleaned = clean_text(text)
    masked = mask_entities(cleaned)
    print(f"\nOriginal:  {text}")
    print(f"Cleaned:   {cleaned}")
    print(f"Masked:    {masked}")

In [ ]:
# Preprocess datasets (uses cache if available)
processed_datasets = {}

for name in ['isot', 'liar']:
    cached = load_processed(name)
    if cached is not None:
        processed_datasets[name] = cached
        print(f"Loaded cached {name}: {cached.shape}")
    else:
        print(f"Preprocessing {name}...")
        processed = preprocess_dataframe(datasets[name])
        save_processed(processed, name)
        processed_datasets[name] = processed
        print(f"Processed {name}: {processed.shape}")

# Show processed sample
print("\n=== Processed Data Sample ===")
print(processed_datasets['isot'][['text', 'cleaned_text', 'processed_text', 'label']].head(3).to_string())

## 4. Feature Engineering

Three parallel pipelines producing a hybrid feature vector:

| Pipeline | Input | Method | Output |
|----------|-------|--------|--------|
| A: Lexical | Masked + lemmatized text | TF-IDF (10K vocab) -> TruncatedSVD | 150 dimensions |
| B: Semantic | Cleaned text | GloVe average embedding | 100 dimensions |
| C: Stylometric | Raw text | 15 hand-crafted features | 15 dimensions |
| **Total** | | | **265 dimensions** |

In [ ]:
from src.feature_engineer import TruthLensFeatureEngine

# Use ISOT as primary training dataset
primary_data = processed_datasets['isot']
train_df, test_df = split_dataset(primary_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# Fit feature engine on training data
engine = TruthLensFeatureEngine()
X_train = engine.fit_transform(
    train_df['processed_text'].tolist(),
    train_df['cleaned_text'].tolist(),
    train_df['text'].tolist(),
)
y_train = train_df['label'].values

# Transform test data
X_test = engine.transform(
    test_df['processed_text'].tolist(),
    test_df['cleaned_text'].tolist(),
    test_df['text'].tolist(),
)
y_test = test_df['label'].values

print(f"\nFeature matrix - Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Feature names ({len(engine.get_feature_names())}): {engine.get_feature_names()[:5]} ... {engine.get_feature_names()[-5:]}")

## 5. Model Training

We train 3 base models with GridSearchCV, then combine them using a Stacking Ensemble:

1. **SVM** (Support Vector Machine) - margin-based, handles high dimensions
2. **Logistic Regression** - interpretable, fast, calibrated probabilities
3. **Random Forest** - captures non-linear relationships
4. **Stacking Ensemble** - LR meta-learner combines base model predictions

In [ ]:
from src.model_trainer import TruthLensTrainer

trainer = TruthLensTrainer()

# Train base models with hyperparameter tuning
trainer.train_base_models(X_train, y_train)

# Train stacking ensemble
trainer.train_ensemble(X_train, y_train)

# Save all models
trainer.save()
print("\nAll models trained and saved.")

## 6. Evaluation

In [ ]:
# Test set evaluation
test_results_df = trainer.evaluate_on_test(X_test, y_test)
print("\n=== Test Set Results ===")
print(test_results_df.to_string(index=False))

In [ ]:
# Confusion Matrices
from src.evaluator import plot_confusion_matrices, plot_roc_curves

plot_confusion_matrices(trainer, X_test, y_test)
plot_roc_curves(trainer, X_test, y_test)

# Display plots inline
from IPython.display import Image as IPImage, display
if os.path.exists(os.path.join(config.PLOTS_DIR, 'confusion_matrices.png')):
    display(IPImage(filename=os.path.join(config.PLOTS_DIR, 'confusion_matrices.png')))
if os.path.exists(os.path.join(config.PLOTS_DIR, 'roc_curves.png')):
    display(IPImage(filename=os.path.join(config.PLOTS_DIR, 'roc_curves.png')))

In [ ]:
# Bias Probe - Can a simple model predict using only entity names?
from src.evaluator import bias_probe

isot_df = processed_datasets['isot']
bias_results = bias_probe(
    isot_df['text'].tolist(),
    isot_df['label'].values,
)
print(f"\nEntity Bias Accuracy: {bias_results.get('entity_accuracy', 'N/A')}")
print(f"Length Bias Accuracy: {bias_results.get('length_accuracy', 'N/A')}")
print(f"Bias Threshold: {config.BIAS_PROBE_THRESHOLD}")

In [ ]:
# Cross-Dataset Evaluation (Train on ISOT, Test on LIAR and vice versa)
from src.evaluator import evaluate_cross_dataset

cross_datasets = {}
for name in ['isot', 'liar']:
    df = processed_datasets[name]
    if len(df) > 5000:
        _, subset = train_test_split(
            df, test_size=5000/len(df),
            stratify=df['label'],
            random_state=config.RANDOM_STATE,
        )
        cross_datasets[name] = subset.reset_index(drop=True)
    else:
        cross_datasets[name] = df

cross_results = evaluate_cross_dataset(trainer, engine, cross_datasets, None)
if cross_results is not None:
    print("\n=== Cross-Dataset Results ===")
    print(cross_results.to_string(index=False))

## 7. Explainability

### LIME (Local Interpretable Model-agnostic Explanations)
Shows which words in the text contributed most to the Fake/Real classification.

### SHAP (SHapley Additive exPlanations)
Shows global feature importance across all predictions.

In [ ]:
from src import explainer as exp_module
import src.preprocessor as preprocessor

# SHAP on Random Forest (TreeExplainer is fast)
if 'RandomForest' in trainer.base_models:
    shap_values, shap_explainer = exp_module.explain_with_shap(
        trainer.base_models['RandomForest'],
        X_test,
        feature_names=engine.get_feature_names(),
    )
    if shap_values is not None:
        exp_module.plot_shap_summary(
            shap_values, X_test,
            feature_names=engine.get_feature_names(),
        )
        # Display inline
        if os.path.exists(os.path.join(config.PLOTS_DIR, 'shap_summary.png')):
            display(IPImage(filename=os.path.join(config.PLOTS_DIR, 'shap_summary.png')))
        
        # Bias Audit
        bias_audit = exp_module.audit_bias(shap_values, engine.get_feature_names())

In [ ]:
# LIME explanation on a sample text
predict_fn = exp_module.create_prediction_explainer(
    engine, preprocessor, trainer.ensemble
)

sample_text = test_df.iloc[0]['text']
sample_label = test_df.iloc[0]['label']
print(f"Sample text (first 200 chars): {sample_text[:200]}...")
print(f"Actual label: {'Fake' if sample_label == 1 else 'Real'}")

explanation = exp_module.explain_with_lime(
    sample_text[:1000],
    predict_fn,
    num_features=10,
    num_samples=500,
)

# Show LIME feature weights
lime_weights = explanation.as_list()
words = [w for w, _ in lime_weights]
weights = [w for _, w in lime_weights]
colors = ['#e74c3c' if w < 0 else '#27ae60' for w in weights]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(words, weights, color=colors)
ax.set_xlabel('Weight (red = Fake, green = Real)')
ax.set_title('LIME: Top Words Influencing Prediction')
plt.tight_layout()
plt.show()

## Summary

### Key Findings
1. **Stacking Ensemble** outperforms all individual models in-domain
2. **Cross-dataset generalization** remains challenging due to fundamental domain mismatch between ISOT (full articles) and LIAR (short claims)
3. **Entity masking + source stripping** reduces bias but does not eliminate it completely
4. **Stylometric features** (sentiment, readability) are the most domain-invariant features

### Next Steps
- Train on combined ISOT+LIAR dataset for better generalization
- Experiment with transformer models (BERT) for semantic understanding
- Add multimodal detection (image + text)
- Support Indian languages for regional fake news detection